# Diabetic Readmission: Missingness, Leakage, and Model Selection

PhDAI 733 · Deliverables 2 and 3 · Sai Teja Bandaru and Rashi Jain

This notebook is a **runner**. All logic lives in `src/` and all parameters in `config.yaml`,
so nothing is defined twice and a change in one place takes effect everywhere.

| module | responsibility |
|---|---|
| `src/data_loader.py` | download, parse, cohort construction, feature engineering |
| `src/preprocessing.py` | the three missingness regimes as leakage-safe pipelines |
| `src/evaluation.py` | diagnostics, cross-validation, tuning, subgroup audit |
| `src/visualization.py` | figures |

Deliverable 2 runs in roughly 10–15 minutes on a Colab CPU instance. Deliverable 3 begins at
section 9 and adds about 30 minutes, almost all of it the randomized search.

## 1. Setup

In [ ]:
# Colab opens this notebook without the rest of the repository, so fetch it.
# Always sync to the latest main: a stale _repo from an earlier run in the same
# session would otherwise shadow any fix that has since been merged.
import os, sys, subprocess
from pathlib import Path

REPO   = "https://github.com/rashijain/diabetic-readmission-audit.git"
BRANCH = "main"

if not Path("src").exists():
    if Path("_repo/.git").exists():
        subprocess.run(["git", "-C", "_repo", "fetch", "--quiet", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", "_repo", "reset", "--hard", "--quiet",
                        f"origin/{BRANCH}"], check=True)
        print("existing clone updated to latest", BRANCH)
    else:
        subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO, "_repo"],
                       check=True)
        print("repository cloned")
    os.chdir("_repo")

sys.path.insert(0, os.getcwd())

# Drop any src modules cached from an earlier run, so the freshly pulled code is used.
for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader   import CFG, load_raw, build_cohort
from src.preprocessing import build_preprocessor, REGIMES
from src.evaluation    import (missingness_diagnostics, run_cv, tune,
                               subgroup_audit, expected_calibration_error, get_models)
from src.visualization import (plot_missingness, plot_leakage_and_regimes,
                               plot_subgroups, plot_decision_curve, net_benefit)

sns.set_theme(style="whitegrid", context="notebook")
SEED = CFG["project"]["random_state"]
Path("outputs").mkdir(exist_ok=True)
print(f"{CFG['project']['name']} | seed {SEED} | regimes {REGIMES}")
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

## 2. Data and cohort

`build_cohort` applies the exclusions and feature engineering defined in `config.yaml`:
expired and hospice discharges removed, zero-variance medication columns dropped, ICD-9
codes collapsed into nine clinical categories, and the medication-burden features derived.

It also resets the index, which later subgroup indexing depends on.

In [ ]:
df = build_cohort(load_raw())
df.head()

## 3. Missingness diagnostics

Tests whether missingness is Missing Not At Random by comparing the readmission rate
where each field is recorded against where it is absent.

In [ ]:
mnar = missingness_diagnostics(df)
display(mnar)
print("\nweight has p = 1.000 - it is the NULL CONTROL, not an MNAR feature.")
print("Its null result is what shows the other four reflect clinical signal.")

In [ ]:
plot_missingness(mnar)

## 4. Leakage ablation (RQ1)

46% of encounters belong to patients appearing more than once, so a naive random split
puts the same patient in train and test. Identical pipelines are run under encounter-level
and patient-grouped cross-validation.

In [ ]:
res_leaky,   _        = run_cv(df, "explicit_mnar", "leaky_encounter")
res_grouped, oof_expl = run_cv(df, "explicit_mnar", "grouped_patient")

leakage = pd.concat([res_leaky, res_grouped], ignore_index=True)
auc = leakage.pivot(index="Model", columns="Split", values="ROC_AUC")
inflation = (auc["leaky_encounter"] - auc["grouped_patient"]).to_frame("ROC_AUC_inflation")
display(inflation.round(4))
print("fold-to-fold SD:", leakage["ROC_SD"].round(4).tolist())
print("\nInflation is far below the noise floor - the leakage hypothesis does not hold here.")

## 5. Missingness regimes (RQ2)

Three preprocessing regimes under patient-grouped validation: drop the high-missingness
features, impute them modally, or encode absence explicitly.

In [ ]:
res_drop,   oof_drop   = run_cv(df, "drop_mnar",       "grouped_patient")
res_impute, oof_impute = run_cv(df, "standard_impute", "grouped_patient")

results = pd.concat([leakage, res_drop, res_impute], ignore_index=True)
results.to_csv("outputs/all_results.csv", index=False)

OOF = {"drop_mnar": oof_drop, "standard_impute": oof_impute, "explicit_mnar": oof_expl}
import pickle
pickle.dump(OOF, open("outputs/oof.pkl", "wb"))

grouped = results[results["Split"] == "grouped_patient"]
display(grouped.pivot(index="Regime", columns="Model", values="PR_AUC")
               .reindex(list(REGIMES)).round(4))

In [ ]:
plot_leakage_and_regimes(results)

## 6. Hyperparameter tuning

Grid search over the selected model under patient-grouped CV. The grid, the split count
and the scoring metric all come from `config.yaml`. The objective is average precision,
not accuracy: the positive class is 11.39% of the cohort, so predicting "no readmission"
throughout would score 88.61% accuracy while identifying nobody.

In [ ]:
grid_search, baseline = tune(df)

## 7. Subgroup audit (RQ3)

In [ ]:
subgroups = subgroup_audit(df, OOF)
by_race = subgroups[(subgroups.Attribute == "race_grp") &
                    (subgroups.Regime == "explicit_mnar")]
display(by_race[["Subgroup", "N", "Prevalence", "ROC_AUC", "ECE", "FNR_at_threshold"]].round(4))

In [ ]:
plot_subgroups(subgroups)

## 8. Clinical utility

Decision curve analysis: net benefit across plausible intervention thresholds, with false
positives penalised at the odds each threshold implies.

In [ ]:
plot_decision_curve(df["target"].values, OOF)

threshold = CFG["validation"]["clinical_decision_threshold"]
for regime in REGIMES:
    nb_value = net_benefit(df["target"].values, OOF[regime]["HistGBDT"],
                           np.array([threshold]))[0]
    print(f"net benefit @ {threshold:.0%}, {regime:16s}: {nb_value:+.5f}")
prevalence = df["target"].mean()
print(f"net benefit @ {threshold:.0%}, {'intervene on all':16s}: "
      f"{prevalence - (1 - prevalence) * (threshold / (1 - threshold)):+.5f}")

## Ethical considerations

- **Potential biases.** `race` is as recorded by hospital staff, not self-identified, so the
  subgroup labels are themselves a product of the documentation process under study.
- **Subgroup performance.** Patients with no recorded race show the lowest discrimination and
  a false negative rate near 0.80 against roughly 0.52 for the best-served group. The model is
  weakest where documentation is thinnest, and that is not randomly distributed.
- **Calibration.** Expected calibration error for Asian and Hispanic patients is roughly ten
  times that for Caucasian patients, so one threshold does not carry one meaning across groups.
- **Interpretability.** Gradient boosting is not interpretable by inspection; a clinician acting
  on a flag is owed a reason. Logistic regression is retained as a transparent reference.
- **Deployment constraint.** Additive only. The model may raise patients onto an intervention
  list; it must never remove a patient a clinician has flagged.
- **Data.** The raw dataset is read only. Exclusions and derived features are applied to an
  in-memory copy; nothing is written back to `diabetic_data.csv`, and it is not committed.

---

# Deliverable 3 — Model Optimization, Real-World Impact, and Ethical Evaluation

Everything above is Deliverable 2 and is unchanged. This section adds the optimization
and the held-out evaluation, in the same runner style: logic in `src/`, parameters in
the `deliverable3` block of `config.yaml`.

| module | responsibility |
|---|---|
| `src/optimization.py` | dev/test split, searches, final fits, operating point, audit |
| `src/d3_figures.py` | Deliverable 3 figures |

**Design.** Patients — not encounters — are split 80/20 first. Every tuning decision,
including the operating threshold, is made on the development side. The held-out set is
scored exactly once, in section 11.

**Runtime.** The randomized search fits 60 configurations across 3 folds, roughly
30 minutes on a Colab CPU instance. It is the slow cell in this notebook.

## 9. Development and held-out split

Splitting on `patient_nbr` rather than on rows matters here: 46% of encounters belong to
patients who appear more than once, so a row-wise split would put a patient's January
stay in development and their March stay in the test set. Deliverable 2 showed that this
does not inflate cross-validated scores on this dataset, but the final estimate is the
one number that has to be beyond question, so the stricter split is used anyway.

In [ ]:
from src.optimization import (split_dev_test, baselines, search_histgbdt, search_grid,
                              tuning_summary, best_params, fit_final, evaluate_holdout,
                              choose_operating_point, operating_stats, capacity_sweep,
                              subgroup_audit_test, feature_importance)
from src.d3_figures import (plot_search, plot_roc_pr, plot_calibration_and_confusion,
                            plot_capacity_and_dca, plot_subgroups_and_importance, STYLE)
import matplotlib.pyplot as plt

D3 = CFG["deliverable3"]
dev, test = split_dev_test(df)

## 10. Hyperparameter tuning

Three untuned baselines first, so there is a bar to clear, then a randomized search over
six gradient-boosting hyperparameters and grid searches for the two reference models.

Random search rather than a grid: the space is uneven, and spending the budget on
sampled points covers it better than an exhaustive grid of the same size
(Bergstra & Bengio, 2012). Objective is average precision, not accuracy — the positive
class is 11.4% of the cohort.

In [ ]:
print("untuned baselines (config.yaml defaults)")
rows = baselines(dev).to_dict("records")

print(f"\nrandomized search, HistGBDT ({D3['n_iter']} candidates)  # ~30 min")
hgb_row, candidates = search_histgbdt(dev)
rows.append(hgb_row)

print("\ngrid searches")
for model in ("LogisticRegression", "RandomForest"):
    rows.append(search_grid(dev, model)[0])

summary = tuning_summary(rows)
params = best_params(summary)
display(summary[["Stage", "Model", "CV_PR_AUC", "SD"]])

In [ ]:
with plt.rc_context(STYLE):
    plot_search(candidates, summary)

In [ ]:
# The gain is real in direction but small against the noise; the spread is the
# finding. Tuning here buys insurance against a bad configuration, not a better one.
default = summary.query("Stage == 'baseline' and Model == 'HistGBDT'").iloc[0]
tuned   = summary.query("Stage == 'randomized_search'").iloc[0]
lo, hi  = candidates.mean_test_score.min(), candidates.mean_test_score.max()

print(f"default {default.CV_PR_AUC:.4f}  ->  tuned {tuned.CV_PR_AUC:.4f}  "
      f"(+{tuned.CV_PR_AUC - default.CV_PR_AUC:.4f}, fold SD {default.SD:.4f})")
print(f"worst candidate {lo:.4f}, best {hi:.4f}  ->  {100*(hi-lo)/hi:.0f}% relative spread")

## 11. Final model evaluation

Four candidates are fitted on the full development set and scored once on the held-out
set: the tuned model, the untuned default, an isotonic-recalibrated variant, and tuned
logistic regression as a transparent reference.

The isotonic variant is calibrated on a patient-disjoint slice of the development data.
Calibrating on the test set would be scoring against the answer sheet.

In [ ]:
models = fit_final(dev, params)
metrics, probs = evaluate_holdout(models, test)
display(metrics)

p_test = probs["Tuned HistGBDT"]
import numpy as np
np.save("outputs/d3_test_probs.npy", p_test)
test[["patient_nbr", "target"] + D3["audit_attributes"]].to_csv(
    "outputs/d3_test_index.csv", index=False)

In [ ]:
with plt.rc_context(STYLE):
    plot_roc_pr(test.target.values, p_test)

## 12. Operating point and clinical utility

Capacity, not probability, defines the decision: the discharge team can follow up a fixed
share of patients. So the cut-off is the matching quantile of **development** out-of-fold
predictions, and is then applied unchanged to the test set. Reading the quantile off the
test set instead would be choosing the operating point with the answer already in hand.

In [ ]:
threshold, dev_oof = choose_operating_point(dev, params)
point = operating_stats(test.target.values, p_test, threshold)
pd.DataFrame([point]).to_csv("outputs/d3_operating_point.csv", index=False)
display(pd.Series(point).to_frame("held-out at the deployed cut-off"))

flag_nobody = 1 - test.target.mean()
print(f"accuracy at this operating point : {point['accuracy']:.4f}")
print(f"accuracy of flagging nobody      : {flag_nobody:.4f}")
print("\nAccuracy is LOWER than the do-nothing baseline while the model is clearly "
      "useful.\nThat is the whole reason average precision, not accuracy, was the "
      "objective throughout.")

In [ ]:
sweep = capacity_sweep(test.target.values, p_test)
display(sweep[["capacity", "precision", "recall", "lift",
               "contacts_per_catch", "F1", "accuracy"]])

In [ ]:
with plt.rc_context(STYLE):
    plot_calibration_and_confusion(test.target.values, p_test, pd.Series(point))
    plot_capacity_and_dca(test.target.values, p_test, sweep)

## 13. Subgroup audit and feature importance

The audit is repeated on the held-out set, which has been used for nothing else, so this
is an out-of-sample check of the equity finding rather than a restatement of it.

In [ ]:
audit = subgroup_audit_test(test, p_test, threshold)
display(audit)

worst = audit[audit.Attribute == "race"].sort_values("FNR").iloc[-1]
best  = audit[audit.Attribute == "race"].sort_values("FNR").iloc[0]
print(f"worst-served: {worst.Group} (n={worst.N:,}) FNR {worst.FNR:.3f}, "
      f"flagged {worst.Flag_rate:.1%}")
print(f"best-served : {best.Group} (n={best.N:,}) FNR {best.FNR:.3f}, "
      f"flagged {best.Flag_rate:.1%}")

In [ ]:
importance = feature_importance(models["Tuned HistGBDT"], test)
with plt.rc_context(STYLE):
    plot_subgroups_and_importance(audit, importance)

## Deliverable 3 — ethical considerations

- **The equity gap replicates out of sample.** Patients with no recorded race are flagged
  at under half the rate of any other group, and roughly nine in ten of their true
  readmissions are missed. The subgroup is small, so the estimate is imprecise, but it
  reproduces the direction found in Deliverable 2 on data that played no part in
  training, tuning or threshold selection.
- **Missingness as signal cuts both ways.** Encoding "not recorded" as information is
  what made the model work, but it means the model learns from clinician ordering
  behaviour. Payer code ranks third in permutation importance. Where testing follows
  access rather than acuity, the model risks encoding a disparity in care as a property
  of the patient (Obermeyer et al., 2019).
- **Accuracy would have hidden all of this.** At the deployed cut-off the model is less
  accurate than flagging nobody. Any evaluation led by accuracy would have rejected a
  useful model, or accepted a useless one.
- **Deployment constraint, unchanged.** Additive only. The model may raise a patient onto
  the follow-up list; it must never remove one a clinician has flagged.
- **Before any clinical use.** Recalibrate on local data, run silently in parallel first,
  and monitor subgroup false negative rates as a gate rather than reporting them once
  (Van Calster et al., 2019).

## Conclusion

The optimization is a genuine but small improvement, and the held-out numbers confirm it
rather than inflate it. Across both deliverables the pattern holds: how the data was
represented mattered roughly seven times more than how the model was configured.

The model earns its place as a way to prioritise a limited follow-up list — it roughly
doubles the yield of a fixed outreach budget — and does not earn the description
"readmission predictor". It misses two thirds of readmissions overall, and far more than
that among the least documented patients.